# pymfx — Flight Statistics & Data Science

This notebook covers the analytical layer of `pymfx`:

1. **`flight_stats(mfx)`** — aggregated metrics with no extra dependencies
2. **`trajectory.to_dataframe()`** — pandas DataFrame with optional event merge
3. **`mfx.to_dict()` / `mfx.to_json()`** — full JSON serialisation

**Requirements:**
```bash
pip install pymfx[ds]   # adds pandas
```

In [ ]:
import sys, math, uuid, json
sys.path.insert(0, '..')  # if running from notebooks/

import pymfx
from pymfx.models import (
    MfxFile, Meta, Trajectory, TrajectoryPoint, SchemaField,
    Events, Event, Index,
)
from pymfx import compute_checksum
from pymfx.stats import flight_stats

print(f"pymfx {pymfx.__version__}")

## Build a demo mission (urban delivery flight)

In [ ]:
BASE_LAT, BASE_LON = 48.8566, 2.3522   # Paris
N = 60
FREQ = 2.0  # Hz

raw_pts = [
    (
        round(i / FREQ, 3),
        round(BASE_LAT + i * 0.00012, 7),
        round(BASE_LON + math.sin(i * 0.18) * 0.0005, 7),
        round(80.0 + math.sin(i * 0.22) * 8, 2),
        round(7.5 + math.cos(i * 0.15) * 2, 2),
        round((i * 6) % 360, 1),
    )
    for i in range(N)
]

schema_fields = [
    SchemaField('t',        'float',   ['no_null']),
    SchemaField('lat',      'float',   ['no_null']),
    SchemaField('lon',      'float',   ['no_null']),
    SchemaField('alt_m',    'float32'),
    SchemaField('speed_ms', 'float32'),
    SchemaField('heading',  'float32'),
]

points    = [TrajectoryPoint(t=r[0], lat=r[1], lon=r[2], alt_m=r[3], speed_ms=r[4], heading=r[5]) for r in raw_pts]
raw_lines = [f"{r[0]:.3f} | {r[1]} | {r[2]} | {r[3]} | {r[4]} | {r[5]}" for r in raw_pts]

event_schema = [
    SchemaField('t',        'float', ['no_null']),
    SchemaField('type',     'str',   ['enum=[takeoff,landing,waypoint,anomaly,rtl]']),
    SchemaField('severity', 'str',   ['enum=[info,warning,critical]']),
    SchemaField('detail',   'str'),
]
event_list = [
    Event(t=0.0,  type='takeoff',  severity='info',    detail='depot_A'),
    Event(t=6.0,  type='waypoint', severity='info',    detail='wp1'),
    Event(t=12.5, type='anomaly',  severity='warning', detail='battery_90pct'),
    Event(t=18.0, type='waypoint', severity='info',    detail='wp2'),
    Event(t=24.0, type='anomaly',  severity='warning', detail='wind_gust'),
    Event(t=29.0, type='landing',  severity='info',    detail='depot_B'),
]
ev_raw = [f"{e.t:.3f} | {e.type} | {e.severity} | {e.detail}" for e in event_list]

lats = [p.lat for p in points];  lons = [p.lon for p in points]
bbox = (round(min(lons), 7), round(min(lats), 7), round(max(lons), 7), round(max(lats), 7))

mfx = MfxFile(
    version='1.0', encoding='UTF-8',
    meta=Meta(
        id=f'uuid:{uuid.uuid4()}',
        drone_id='UrbanFly-X1-SN778899',
        drone_type='multirotor',
        pilot_id='FR-PILOT-2024',
        date_start='2026-09-01T14:00:00Z',
        date_end='2026-09-01T14:00:30Z',
        duration_s=29.5,
        status='complete',
        application='delivery',
        location='Paris, France',
        sensors=['rgb', 'lidar'],
        data_level='raw',
        license='CC-BY-4.0',
        contact='ops@urbanfly.fr',
    ),
    trajectory=Trajectory(
        frequency_hz=FREQ,
        schema_fields=schema_fields,
        points=points,
        checksum=compute_checksum(raw_lines),
        raw_lines=raw_lines,
    ),
    events=Events(
        schema_fields=event_schema,
        events=event_list,
        checksum=compute_checksum(ev_raw),
        raw_lines=ev_raw,
    ),
    index=Index(bbox=bbox, anomalies=2),
)

print(pymfx.validate(mfx))
print(f"{N} points · {len(event_list)} events · {N/FREQ:.1f}s")

## 1. Flight statistics

`flight_stats()` returns a `FlightStats` dataclass computed using only the Python standard library (Haversine great-circle distance, pure math).

In [ ]:
stats = flight_stats(mfx)
print(stats)

In [ ]:
# Access individual fields
print(f"Duration  : {stats.duration_s:.1f} s")
print(f"Distance  : {stats.total_distance_m:.0f} m  ({stats.total_distance_km:.3f} km)")
print(f"Altitude  : {stats.alt_min_m:.1f} – {stats.alt_max_m:.1f} m  (mean {stats.alt_mean_m:.1f} m)")
print(f"Speed     : max {stats.speed_max_ms:.1f} m/s  mean {stats.speed_mean_ms:.1f} m/s")
print(f"Points    : {stats.point_count}")

In [ ]:
# Also accessible via the public API
stats2 = pymfx.flight_stats(mfx)
assert stats2.duration_s == stats.duration_s
print("Public API: pymfx.flight_stats() ✓")

## 2. Pandas DataFrame

`trajectory.to_dataframe()` converts the trajectory points to a `pandas.DataFrame`.
Pass `events=mfx.events` to merge event information using a nearest-time join.

In [ ]:
# Trajectory-only DataFrame
df = mfx.trajectory.to_dataframe()
print(df.shape)
df.head()

In [ ]:
# With events merged via nearest-time join
df_ev = mfx.trajectory.to_dataframe(events=mfx.events)
print(df_ev.columns.tolist())
df_ev[df_ev['event_type'].notna()]

In [ ]:
# Standard pandas operations
print("Describe:")
df[['alt_m', 'speed_ms']].describe().round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(df['t'], df['alt_m'],    color='#1a73e8', linewidth=1.8)
axes[0].set_ylabel('Altitude (m)');  axes[0].grid(alpha=0.3)

axes[1].plot(df['t'], df['speed_ms'], color='#e8711a', linewidth=1.8)
axes[1].set_ylabel('Speed (m/s)');   axes[1].set_xlabel('Time (s)');  axes[1].grid(alpha=0.3)

# Mark events
for _, row in df_ev[df_ev['event_type'].notna()].iterrows():
    for ax in axes:
        ax.axvline(row['t'], color='orange', linestyle='--', alpha=0.6)

fig.suptitle(f"{mfx.meta.drone_id} — Altitude & Speed from DataFrame", fontweight='bold')
fig.tight_layout()
plt.show()

## 3. JSON serialisation

`mfx.to_dict()` converts the entire `MfxFile` (all nested dataclasses) to a plain Python `dict`.
`mfx.to_json()` returns a JSON string.

In [ ]:
d = mfx.to_dict()

print("Top-level keys:", list(d.keys()))
print("Meta keys     :", list(d['meta'].keys()))
print("First point   :", d['trajectory']['points'][0])

In [ ]:
json_str = mfx.to_json(indent=2)

# Preview
print(json_str[:500], "...")

In [ ]:
# Round-trip: JSON → dict → check values
reloaded = json.loads(json_str)
assert reloaded['meta']['drone_id'] == mfx.meta.drone_id
assert len(reloaded['trajectory']['points']) == N
print("Round-trip JSON ✓")

In [ ]:
# Save as JSON
with open('/tmp/flight_export.json', 'w') as f:
    f.write(json_str)
print(f"Saved {len(json_str):,} characters to /tmp/flight_export.json")

## Summary

| Feature | Function | Returns | Extra dep |
|---|---|---|---|
| Aggregated metrics | `pymfx.flight_stats(mfx)` | `FlightStats` | none |
| Trajectory DataFrame | `mfx.trajectory.to_dataframe()` | `pd.DataFrame` | pandas |
| DataFrame + events | `mfx.trajectory.to_dataframe(events=mfx.events)` | `pd.DataFrame` | pandas |
| Dict serialisation | `mfx.to_dict()` | `dict` | none |
| JSON serialisation | `mfx.to_json(indent=2)` | `str` | none |